## 3.3 在Agent运行图中访问长期记忆

我们可以在工具或中间件中访问长期记忆。

### 3.3.1 在工具中访问长期记忆
① 基于InMemoryStore

In [12]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="deepseek-v4.1-flash",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [13]:

from langchain_protocol import NotRequired
from langgraph.prebuilt import ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState

store = InMemoryStore()

# 自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id : NotRequired[str]

# 保存用户信息到长期记忆中
@tool(parse_docstring=True)
def save_user_info(name : str , runtime : ToolRuntime) -> str:
    """
    将客户信息保存在长期记忆中

    Args:
        name : 用户名
        runtime : 工具的运行时

    Returns:
        str : 保存状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name" : name}

    runtime.store.put(namespace, key, value)
    return "saved"


@tool(parse_docstring=True)
def get_user_info(runtime : ToolRuntime) -> str:
    """
    从长期记忆中读取客户的信息

    Args:
        runtime : 工具的运行时上下文

    Returns:
        str : 用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]

    item = runtime.store.get(namespace,key)
    return str(item.value) if item else "unknown"

agent = create_agent(
    model=model,
    tools=[save_user_info,get_user_info],
    store=store,
    state_schema=CustomState,
    system_prompt="用户提及个人信息时，可以使用工具保存用户信息。如果用户询问个人信息时，可以尝试使用工具读取用户信息"
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()


============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你 😊 让我先把你的名字记下来。
Tool Calls:
  save_user_info (call_00_eHGU8dmHPR8oQ4XgeRkG6156)
 Call ID: call_00_eHGU8dmHPR8oQ4XgeRkG6156
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

已经记住啦，小花！以后我们聊天我都能记得你的名字 😊

有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_t07LTDKsrbk3l5lTvjqd0850)
 Call ID: call_00_t07LTDKsrbk3l5lTvjqd0850
  Arg

**Agent 使用长期记忆的步骤**

上面的例子里，两次 `invoke` 是两个互不相干的会话：没有配置 checkpointer，第二次 invoke 看不到第一次的消息。第二次仍能答出“小花”，靠的是长期记忆。整个过程分五步：

![Agent 使用长期记忆的五个步骤](assets/13-agent-long-memory-flow.svg)

1. **把 store 交给 Agent**：`create_agent(..., store=store)`。
2. **告诉 Agent 当前用户是谁**：invoke 时传入 `user_id`，它存在 state 里。
3. **模型决定调用哪个工具**：模型只能填写 `name` 这类普通参数，看不到 `runtime`。
4. **tools 节点注入 runtime，工具读写 store**：工具通过 `runtime.state` 拿到 user_id，通过 `runtime.store` 读写长期记忆。
5. **工具结果交回模型，跨会话读取**：同一个 store、同一个 namespace + key，任何会话都能读到。

> 先回顾 Chapter07 `01-Agent的基本用法` 2.4 节：`create_agent` 返回的 Agent 内部是一张“图”。图由**节点**组成，每个节点是一个函数：`model` 节点调用模型，`tools` 节点执行工具。节点之间传递的数据叫 **state**（状态），默认只有 `messages`。下面会反复用到这几个词。

**第一步：把 store 交给 Agent**

`create_agent` 收到 `store` 后自己并不使用，而是在最后编译图时交给图：

```python
return graph.compile(
    checkpointer=checkpointer,
    store=store,
    ...
)
```

每次 `agent.invoke(...)`，LangGraph 都会为这次运行创建一个 **`Runtime` 对象**（运行时），把 store 放进去：

```python
runtime = Runtime(
    context=...,          # invoke 时传入的 context（需要配置 context_schema）
    store=store,          # 编译时传入的 store
    stream_writer=...,
    ...
)
```

这个 `Runtime` 会交给图里的每个节点，所以工具和中间件拿到的都是**同一个** store 对象：工具里写 `runtime.store`，中间件的钩子函数也会收到 `runtime` 参数。Agent 写入的数据，就在我们自己创建的 `store` 变量里（第五步会验证）。

**第二步：告诉 Agent 当前用户是谁**

长期记忆要按用户存取，工具需要知道用哪个 key。例子把 user_id 放在 state 里传进去：

其中，CustomState扩展了 Agent 的标准状态。除了默认的 messages （历史消息列表）之外，还额外增加了一个 user_id 字段。这样，Agent 在运行过程中随时随地都能知道当前和它说话的用户 ID 是什么。

具体过程：

* `CustomState` 继承 `AgentState`，多声明了一个字段 `user_id`，`NotRequired` 表示这个字段可以不传。
* `create_agent(..., state_schema=CustomState)` 让 Agent 使用这个状态结构。
* `invoke` 传入的字典里，`messages` 和 `user_id` 两个键都会成为 state 的一部分。
* 工具里用 `runtime.state["user_id"]` 读取它，作为 store 的 key：`runtime.store.put(("users",), key, value)`。

两次 invoke 传入的 `user_id` 相同，所以读写的是同一条长期记忆。

**第三步：模型决定调用哪个工具——为什么 AIMessage 的工具参数里没有 runtime**

上面的输出里，`save_user_info` 的 `Args` 只有 `name: 小花`，`get_user_info` 的 `Args` 是空的，都没有 `runtime`。原因是 **runtime 不是模型填写的参数**：工具发给模型时它被去掉了，执行前再由 tools 节点补进去。

![为什么 AIMessage 的工具参数里没有 runtime](assets/14-tool-runtime-injection.svg)

一个工具有两份参数说明：

| | 内容 | 用途 |
|---|---|---|
| `args_schema` | 函数的全部参数：`name`、`runtime` | 执行工具时校验参数 |
| `tool_call_schema` | 去掉“注入参数”后剩下的参数：只有 `name` | 发给模型 |

生成 `tool_call_schema` 时会逐个检查参数类型，用 `_is_injected_arg_type` 判断是不是注入参数，是就跳过：

```python
for name, type_ in get_all_basemodel_annotations(full_schema).items():
    if not _is_injected_arg_type(type_):
        fields.append(name)
```

`ToolRuntime` 的父类是 `_DirectlyInjectedToolArg`，所以只要参数类型写成 `ToolRuntime`，就会被认定为注入参数。模型只能生成它见过的参数，所以 AIMessage 的 `tool_calls` 里只有 `name`。`get_user_info` 只有 `runtime` 一个参数，去掉后就没有可填的参数了。docstring 里 `runtime : 工具的运行时` 这行说明也不会发给模型。

下面查看两份参数说明，以及真正发给模型的工具定义（只做转换，不调用模型）：

In [ ]:
import json
from langchain_core.utils.function_calling import convert_to_openai_tool

for t in [save_user_info, get_user_info]:
    print(t.name)
    print("  args_schema 的字段     :", list(t.args_schema.model_fields))
    print("  tool_call_schema 的字段:", list(t.tool_call_schema.model_fields))

# ChatOpenAI.bind_tools 内部就是用 convert_to_openai_tool 把工具转成这种格式发给模型
print(json.dumps(convert_to_openai_tool(save_user_info), ensure_ascii=False, indent=1))

save_user_info
  args_schema 的字段     : ['name', 'runtime']
  tool_call_schema 的字段: ['name']
get_user_info
  args_schema 的字段     : ['runtime']
  tool_call_schema 的字段: []
{
 "type": "function",
 "function": {
  "name": "save_user_info",
  "description": "将客户信息保存在长期记忆中",
  "parameters": {
   "properties": {
    "name": {
     "description": "用户名",
     "type": "string"
    }
   },
   "required": [
    "name"
   ],
   "type": "object"
  }
 }
}


**第四步：tools 节点注入 runtime，工具读写 store**

模型返回带 `tool_calls` 的 AIMessage 后，Agent 进入 `tools` 节点，实现它的类是 `ToolNode`。对每个工具调用，它做三件事：

1. 新建一个 `ToolRuntime`，把这次运行的信息装进去：

```python
tool_runtime = ToolRuntime(
    state=state,                  # 当前 state，例子里是 {"messages": [...], "user_id": "user-1"}
    tool_call_id=call["id"],
    config=cfg,
    context=runtime.context,      # 来自第一步的 Runtime
    store=runtime.store,          # 来自第一步的 Runtime，就是传给 create_agent 的 store
    stream_writer=runtime.stream_writer,
    ...
)
```

2. 用 `_inject_tool_args` 把模型给的参数**复制一份**，在副本里放进 runtime：

```python
stripped_args = {k: v for k, v in tool_call_copy["args"].items()
                 if k not in injected.all_injected_keys}    # 先删掉模型自己填的同名参数
tool_call_copy["args"] = {**stripped_args, **injected_args}  # 再放进真正的 runtime
```

3. 用这份副本调用 `tool.invoke(...)`，按 `args_schema` 校验时 `name` 和 `runtime` 都齐了。

注入只发生在副本上，messages 里保存的 AIMessage 仍是模型的原始输出，所以 `response` 里永远看不到 runtime。

工具拿到的 `ToolRuntime` 常用字段：

| 字段 | 内容 | 例子里的用法 |
|---|---|---|
| `state` | 当前 state | `runtime.state["user_id"]` 取用户 ID |
| `store` | 传给 `create_agent` 的 store | `runtime.store.put(...)` / `runtime.store.get(...)` |
| `context` | invoke 时传入的 context（需要配置 `context_schema`） | 未使用 |
| `config` | 本次运行的配置，如 `thread_id` | 未使用 |
| `tool_call_id` | 这次工具调用的 ID，即 AIMessage 里的 `id` | 未使用 |
| `stream_writer` | 用来流式输出自定义数据 | 未使用 |

为什么这样设计：

* `ToolRuntime` 是 Python 对象，里面有 store 和整份 state，而模型只能输出 JSON，生成不了它。
* 读哪个用户、写进哪个 store，不能交给模型决定。第 2 步先删掉模型填的同名参数，源码注释写明目的是防止 LLM 伪造这些隐藏参数。

下面用一个按顺序返回预设消息的假模型验证（不调用 API）。假模型故意在工具参数里多填一个 `runtime`，看工具实际收到什么：

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage


class ScriptedModel(FakeMessagesListChatModel):
    """按顺序返回预先写好的消息的假模型，不调用 API"""

    def bind_tools(self, tools, **kwargs):
        return self


@tool
def show_runtime(name: str, runtime: ToolRuntime) -> str:
    """打印工具实际收到的参数"""
    print("工具收到的 name    :", name)
    print("工具收到的 runtime :", type(runtime).__name__)
    print("  runtime.state 的键         :", list(runtime.state))
    print("  runtime.state['user_id']   :", runtime.state["user_id"])
    print("  runtime.store is demo_store:", runtime.store is demo_store)
    print("  runtime.tool_call_id       :", runtime.tool_call_id)
    return "ok"


demo_store = InMemoryStore()
fake_model = ScriptedModel(responses=[
    # 故意在工具参数里多填一个 runtime，验证 tools 节点会丢掉它
    AIMessage(content="", tool_calls=[{
        "name": "show_runtime",
        "args": {"name": "小花", "runtime": "模型伪造的 runtime"},
        "id": "call_1",
    }]),
    AIMessage(content="完成"),
])
demo_agent = create_agent(
    model=fake_model,
    tools=[show_runtime],
    store=demo_store,
    state_schema=CustomState,
)
result = demo_agent.invoke({"messages": [HumanMessage("我是小花")], "user_id": "user-1"})
print("AIMessage 里的工具参数:", result["messages"][1].tool_calls[0]["args"])

工具收到的 name    : 小花
工具收到的 runtime : ToolRuntime
  runtime.state 的键         : ['messages', 'user_id']
  runtime.state['user_id']   : user-1
  runtime.store is demo_store: True
  runtime.tool_call_id       : call_1
AIMessage 里的工具参数: {'name': '小花', 'runtime': '模型伪造的 runtime'}


从输出可以看到：

* 工具收到的 `runtime` 是真正的 `ToolRuntime`，模型伪造的字符串被丢掉了；`runtime.store` 就是传给 `create_agent` 的 `demo_store`，`runtime.state` 里有 invoke 传入的 `user_id`。
* AIMessage 里的参数仍是模型的原样输出（包括伪造的值），说明注入没有改动 messages。正常情况下模型看不到 runtime，不会生成它，这里是故意构造的。

**第五步：工具结果交回模型，跨会话读取**

工具的返回值会被包装成 `ToolMessage` 追加到 messages，Agent 回到 `model` 节点，模型看到工具结果后生成最终回复（第一个会话里的“已经记住啦”）。

第二个会话是一次新的 invoke：没有 checkpointer，也没有 thread_id，消息从零开始，模型并不知道“小花”。它调用 `get_user_info`，工具用同一个 user_id 作为 key，从同一个 store 读出 `{'name': '小花'}`。

| | 短期记忆 | 长期记忆 |
|---|---|---|
| 存在哪里 | checkpointer | store |
| 按什么区分 | `thread_id`，一个会话一份 | namespace + key，由我们自己决定 |
| 本例 | 没有配置，两次 invoke 互不相通 | 两次 invoke 都读写 `("users",)` / `"user-1"` |

Agent 写入的记录，就在我们自己创建的 `store` 里：

In [ ]:
# 第一个会话写入的记录，就在我们自己创建的 store 里
for item in store.search(("users",)):
    print(item.namespace, item.key, item.value)

('users',) user-1 {'name': '小花'}


2. 基于SqliteStore

In [14]:
from pathlib import Path
db_path = Path(
    "/Users/skk/Developer/lang-chain-learn/Chapter09_memory/data/memory.sqlite"
)

In [16]:
from pathlib import Path
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.store.sqlite import SqliteStore
from rich import print as rprint

# 自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id : NotRequired[str]

# 保存用户信息到长期记忆中
@tool(parse_docstring=True)
def save_user_info(name : str , runtime : ToolRuntime) -> str:
    """
    将客户信息保存在长期记忆中

    Args:
        name : 用户名
        runtime : 工具的运行时

    Returns:
        str : 保存状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name" : name}

    runtime.store.put(namespace, key, value)
    return "saved"


@tool(parse_docstring=True)
def get_user_info(runtime : ToolRuntime) -> str:
    """
    从长期记忆中读取客户的信息

    Args:
        runtime : 工具的运行时

    Returns:
        str : 用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]

    item = runtime.store.get(namespace,key)
    return str(item.value) if item else "unknown"

with SqliteStore.from_conn_string(db_path) as store:
    agent = create_agent(
        model=model,
        tools=[save_user_info,get_user_info],
        store=store,
        state_schema=CustomState,
        system_prompt="用户提及个人信息时，可以使用工具保存用户信息。如果用户询问个人信息时，可以尝试使用工具读取用户信息"
    )

    print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
    response1 = agent.invoke({
        "messages": [HumanMessage("你好，很高兴认识你，我是小名")],
        "user_id": "user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    response2 = agent.invoke({
        "messages": [HumanMessage("我是谁")],
        "user_id": "user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小名
================================== Ai Message ==================================

你好，小名！很高兴认识你，有什么可以帮你的吗？

让我先把你的信息记下来。
Tool Calls:
  save_user_info (call_00_SYeX79CzCvVJbqSAfX0b4978)
 Call ID: call_00_SYeX79CzCvVJbqSAfX0b4978
  Args:
    name: 小名
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

已经帮你记好啦，小名！😊

以后有需要随时找我，有什么问题都可以问我～
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_g1CUMbiHah2c7cbSox0Y2642)
 Call ID: call_00_g1CUMbiHah2c7cbSox0Y2642

In [18]:
rprint(response1)

{
    'messages': [
        HumanMessage(
            content='你好，很高兴认识你，我是小名',
            additional_kwargs={},
            response_metadata={},
            id='756f5af6-7ccb-4e37-998d-4c452a20153c'
        ),
        AIMessage(
            content='你好，小名！很高兴认识你，有什么可以帮你的吗？\n\n让我先把你的信息记下来。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 83,
                    'prompt_tokens': 362,
                    'total_tokens': 445,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 22,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None,
                        'cached_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None,
                        'accepted_prediction_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'cache_creation_input_tokens': 0,
                    'cache_read_input_tokens': 0,
                    'cached_tokens': 0,
                    'completion_thinking_tokens': 22,
                    'credit': 0.01,
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 362,
                    'prompt_cache_write_tokens': 0
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-v4.1-flash',
                'system_fingerprint': None,
                'id': '21f713979ab34a163c50446fb03f1902',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a11c49-75d0-71e0-a6cd-c3a556159c70-0',
            tool_calls=[
                {
                    'name': 'save_user_info',
                    'args': {'name': '小名'},
                    'id': 'call_00_SYeX79CzCvVJbqSAfX0b4978',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 362,
                'output_tokens': 83,
                'total_tokens': 445,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 22}
            }
        ),
        ToolMessage(
            content='saved',
            name='save_user_info',
            id='34c1b84f-465d-445f-af75-f5e3ba746f42',
            tool_call_id='call_00_SYeX79CzCvVJbqSAfX0b4978'
        ),
        AIMessage(
            content='已经帮你记好啦，小名！😊\n\n以后有需要随时找我，有什么问题都可以问我～',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 37,
                    'prompt_tokens': 438,
                    'total_tokens': 475,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 14,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None,
                        'cached_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None,
                        'accepted_prediction_tokens': 0,
                        'r

**SqliteStore 版本**

五个步骤完全相同，区别只在第一步交给 Agent 的 store：

* InMemoryStore 的数据在内存里，内核重启就没了；SqliteStore 写进 `data/memory.sqlite` 文件，重启后还在。
* `with SqliteStore.from_conn_string(db_path) as store:` 结束时会关闭数据库连接，所以 agent 要在 `with` 块里使用。
* 这里没有调用 `store.setup()` 也能运行：SqliteStore 每次执行 SQL 前会检查是否已经 setup，没有就自动建表（`_cursor()` 里的 `if not self.is_setup: self.setup()`）。

下面新开一个连接读取文件，上面 agent 写入的记录还在：

In [ ]:
# 新开一个连接读取 memory.sqlite：agent 写入的记录已经保存在文件里
with SqliteStore.from_conn_string(db_path) as s:
    item = s.get(("users",), "user-1")
    print(item.namespace, item.key, item.value)

('users',) user-1 {'name': '小名'}


### 3.3.2 在中间件访问长期记忆

① Node-style hooks中访问

以 before_model 为例，其钩子函数签名如下:

```python
def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
```

函数中存在runtime，因此可以通过runtime访问

② Wrap-style hooks中访问

wrap_model_call 函数签名如下:
```python
def wrap_model_call(
self,
request: ModelRequest[ContextT],
handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]:
```

ModelRequest内部属性存在runtime